# audiollm: from data to LLM tokens, step by step

This notebook walks through the pipeline in [`train.py`](train.py) and **reuses its functions**, so what you see here
is exactly what the training loop gets.

| § | what |
|---|---|
| 0 | **Setup and parameters**: every knob, and which trained run is reused |
| 1 | **The data**: the `TRAIN_SPLIT` mix, val / test splits, durations, a sample utterance |
| 2 | **Split rules and batching**: train / val / test selection, length-bucketed batches |
| 3 | **Audio features**: wav → STFT → mel → log → MVN → pad → stack `STACK` frames = 1 "patch" |
| 4 | **Choosing the front-end**: patch size vs LLM width (4a), side-by-side `SR / N_FFT / HOP / N_MELS / STACK` sweep (4b) |
| 5 | **The projector**: audio patches → LLM token embeddings, what it learned, where its tokens land |
| 6 | **The final input sequence**: `[BOS][audio…]["Transcription:"][text][EOS]`, padding, attention mask, labels |
| 7 | *(optional)* a forward pass + greedy decode with the trained checkpoint, and where LoRA sits in the LLM |
| 8 | **Run tracking**: `runs/latest`, `runs/INDEX.md`, the per-run `plots.png` |

For a layer-by-layer look at how the **conv projector** turns mel frames into one token, see
[`projector_anatomy.ipynb`](projector_anatomy.ipynb).

```
wav ─► N_MELS-d log-mel (N_FFT win / HOP hop) ─► MVN ─► stack STACK frames = 1 patch (STACK·N_MELS-d)
    ─► projector (linear | mlp | conv) ─► [BOS][audio…]["Transcription:"][ text ][EOS] ─► LFM2.5 (frozen) + LoRA
```
(`train.py` defaults: 80 mels, 25 ms / 10 ms, `STACK = 16` → 1280-d patches at 6.25 Hz, `PROJECTOR = "linear"`.
The strongest runs so far use `PROJECTOR = "conv"`, `STACK = 8` (12.5 Hz, the paper's rate); put those in `OVERRIDES`
to walk through them. Trained: the projector (fully) and LoRA adapters on the LLM's linear layers; the LLM weights stay
frozen.)

**Every knob is in the `PARAMETERS` block of the first code cell**: edit it, then *Run All*.
The LLM parts run on the **GPU** (`DEVICE = "auto"`, bf16 like training); set `DEVICE = "cpu"` if a training run needs
the whole GPU. Feature extraction, the projector and the plots are always on CPU.

## 0. Setup and parameters

`OVERRIDES` accepts any UPPERCASE constant of `train.py` (same names as its CLI flags, upper-cased), so the notebook shows
exactly what `python train.py --stack 16 ...` would feed the model. A trained run is only reused if it was trained with
the same patch shape and projector (`MODEL`, `SR`, `N_FFT`, `HOP`, `N_MELS`, `STACK`, `PROJECTOR`); otherwise the
projector is randomly initialised.


In [ ]:
import json, os, random, sys
from collections import Counter
from pathlib import Path

# train.py uses paths relative to the repo root (data/LibriSpeech, runs/...)
ROOT = Path.cwd()
while not (ROOT / "train.py").exists():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display
from transformers import AutoConfig

import train as T   # the training script: all functions + config constants (MODEL = LFM2.5-230M-Base)

# ═══════════════════════════ PARAMETERS: edit, then Run All ═══════════════════════════
OVERRIDES = {
    # "STACK": 16,       # frames per patch: 8 -> 640-d @ 12.5 Hz, 16 -> 1280-d @ 6.25 Hz (230M hidden_size = 1024)
    # "N_MELS": 128,     # mel bins per frame (MVN stats for a new value are computed + cached in section 3c, ~1 min)
    # "HOP": 160, "N_FFT": 400,
    # "MODEL": "LiquidAI/LFM2.5-1.2B-Base",
    # "PROJECTOR": "conv", "STACK": 8,   # the 544 h mix runs (strided-conv projector, 12.5 Hz tokens)
}
USE_TRAINED_RUN = True                     # reuse the newest finished run with the same patch shape (if any)
SAMPLE_SPLIT = "train-clean-100"           # split of the utterance walked through in sections 1 and 3
SAMPLE_INDEX = 42                          # its index in that split
STACK_SWEEP = [4, 6, 8, 10, 12, 16, 20, 24, 32]   # patch sizes compared against the LLM width in section 4
FRONTEND_SWEEP = [                         # front-end configs compared side by side in section 4b (missing keys = current value)
    {},                                    # the current config
    {"STACK": 8},                          # paper rate: 12.5 Hz tokens, 640-d patches
    {"STACK": 16},                         # train.py default: 6.25 Hz tokens, 1280-d patches
    {"N_MELS": 128},                       # finer frequency resolution, 2048-d patches
    {"N_MELS": 40, "STACK": 32},           # coarse frequency, same 1280-d patch, 3.1 Hz tokens
    {"N_FFT": 512},                        # 32 ms window: finer FFT bins, blurrier onsets
    {"HOP": 320, "N_FFT": 640, "STACK": 8},  # 20 ms frames, same 160 ms per token as STACK=16
    {"SR": 8000, "N_FFT": 200, "HOP": 80},   # telephone band (drops everything above 4 kHz)
]
FRONTEND_PROBE_N = 200                     # train utterances each config in section 4b is measured on
DEVICE = "auto"                            # "auto" = GPU if available (ROCm: the AMD GPU is "cuda"), "cuda", "cpu"
# ════════════════════════════════════════════════════════════════════════════════════════

unknown = [k for k in OVERRIDES if k not in T.DEFAULT_CONFIG]
assert not unknown, f"not train.py constants: {unknown}"
T.apply_config(OVERRIDES)

# A checkpoint is only reusable if everything that defines a patch matches
SHAPE_KEYS = ["MODEL", "SR", "N_FFT", "HOP", "N_MELS", "STACK", "PROJECTOR"]
wanted = {k: getattr(T, k) for k in SHAPE_KEYS}
RUN_DIR = None
if USE_TRAINED_RUN:
    finished = sorted(Path("runs").glob("*/proj.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
    for proj_file in finished:
        run_cfg = json.load(open(proj_file.parent / "config.json"))
        if all(run_cfg.get(k, T.DEFAULT_CONFIG[k]) == v for k, v in wanted.items()):   # missing key = older run, default
            RUN_DIR = proj_file.parent
            run_cfg.setdefault("PRECISION", "fp32")   # runs from before PRECISION existed
            T.apply_config({**run_cfg, **OVERRIDES})  # the run's config, explicit overrides still win
            break
print("trained run:", RUN_DIR or f"none with {wanted} (projector will be randomly initialised)")

# Notebook device: GPU -> bf16 like training; CPU -> fp32 (CPU bf16 is slow)
if DEVICE == "auto":
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
T.DEVICE, T.DTYPE = DEVICE, (torch.float32 if DEVICE == "cpu" else torch.bfloat16)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| dtype:", T.DTYPE)
torch.set_num_threads(4)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})

FPS = T.SR / T.HOP                  # mel frames per second
PATCH_DIM = T.STACK * T.N_MELS      # numbers per audio token
H = AutoConfig.from_pretrained(T.MODEL).hidden_size
changed = {k: getattr(T, k) for k in T.DEFAULT_CONFIG if k != "DTYPE" and getattr(T, k) != T.DEFAULT_CONFIG[k]}
print("differs from train.py defaults:", changed or "nothing")
print(f"MODEL={T.MODEL}  SR={T.SR}  N_FFT={T.N_FFT}  HOP={T.HOP}  N_MELS={T.N_MELS}  STACK={T.STACK}  PROJECTOR={T.PROJECTOR}")
print(f"frame rate = {FPS:.0f} Hz, audio token rate = {FPS / T.STACK:.2f} Hz ({1000 * T.STACK / FPS:.0f} ms/token), "
      f"patch dim = {T.STACK} x {T.N_MELS} = {PATCH_DIM} -> hidden_size {H} ({PATCH_DIM / H:.2f}x)")


## 1. The data

Training reads `TRAIN_SPLIT`, a `+`-joined mix of splits (default: `train-clean-100 + train-clean-360 + ami_ihm/train +
fleurs_en/train`, ~544 h):

* **LibriSpeech** names (`train-clean-100`) are unpacked by `prepare_data.py` in the OpenSLR layout:
  `data/LibriSpeech/<split>/<speaker>/<chapter>/<utt-id>.flac` + one `<speaker>-<chapter>.trans.txt` per chapter,
  scanned once and cached as `manifest.jsonl`.
* **`<corpus>/<split>`** names (`ami_ihm/train`) are unpacked by `unpack_parquet_asr_corpora.py` into
  `data/asr_corpora/<corpus>/<split>/` with a `manifest.jsonl`.

`T.load_split()` returns one dict per utterance: `id, path, text, sec`, with the text normalised by
`T.normalize_training_text()` (lowercase ASCII, annotation tags removed).

In [ ]:
!find data/LibriSpeech/train-clean-100 -maxdepth 3 | head -8
first_trans = sorted(Path("data/LibriSpeech/train-clean-100").rglob("*.trans.txt"))[0]
print("\n", first_trans); print("".join(first_trans.read_text().splitlines(True)[:3]))

In [ ]:
# LibriSpeech splits are scanned once (sf.info on every file header), then read from their manifest.jsonl
TRAIN_PARTS = T.unweighted_split(T.TRAIN_SPLIT).split("+")      # "*K" repeat weights are dropped here
splits = {name: T.load_split(name) for name in [*TRAIN_PARTS, T.VAL_SPLIT, *T.TEST_SPLITS]}

def speaker(item): return item["id"].split("-")[0]   # LibriSpeech ids are <speaker>-<chapter>-<utt>

print(f"{'split':<17}{'utts':>7}{'hours':>8}{'speakers':>10}{'mean s':>8}{'max s':>7}{'words/utt':>11}")
for name, items in splits.items():
    secs = np.array([it["sec"] for it in items])
    words = np.mean([len(it["text"].split()) for it in items])
    n_spk = "-" if "/" in name else len({speaker(it) for it in items})   # other corpora use other id schemes
    print(f"{name:<17}{len(items):>7}{secs.sum() / 3600:>8.1f}{n_spk:>10}"
          f"{secs.mean():>8.1f}{secs.max():>7.1f}{words:>11.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for name, items in splits.items():
    axes[0].hist([it["sec"] for it in items], bins=70, range=(0, 36), histtype="step", density=True, lw=1.5, label=name)
for limit, name in [(T.MIN_TRAIN_SEC, "MIN"), (T.MAX_TRAIN_SEC, "MAX")]:
    axes[0].axvline(limit, color="k", ls="--", lw=1, label=f"{name}_TRAIN_SEC={limit:g}")
axes[0].set(xlabel="duration (s)", ylabel="density", title="Utterance duration per split"); axes[0].legend(fontsize=8)

train_secs = np.array([it["sec"] for name in TRAIN_PARTS for it in splits[name]])
n_tokens = np.ceil(train_secs * T.SR / T.HOP / T.STACK)   # ~ audio tokens the LLM sees per utterance
axes[1].hist(n_tokens, bins=60, color="C0")
axes[1].set(xlabel="audio tokens per utterance", ylabel="count", title="train mix: sequence length in audio tokens")
plt.tight_layout(); plt.show()

### A sample utterance

In [ ]:
sample = splits[SAMPLE_SPLIT][SAMPLE_INDEX]
wav, sr = sf.read(sample["path"], dtype="float32")
print(sample["id"], f"| {sample['sec']:.2f} s | {len(wav)} samples @ {sr} Hz")
print("text:", sample["text"])
display(Audio(wav, rate=sr))

t = np.arange(len(wav)) / sr
plt.figure(figsize=(13, 2.2)); plt.plot(t, wav, lw=0.4)
plt.xlim(0, t[-1]); plt.xlabel("time (s)"); plt.title("waveform"); plt.show()

## 2. How the data is split

| role | source | rule (in `train.train()`) |
|---|---|---|
| **train** | the `TRAIN_SPLIT` mix (`"<split>*K"` repeats a split K times per epoch) | `MIN_TRAIN_SEC <= sec <= MAX_TRAIN_SEC`; `TRAIN_N > 0` keeps a seeded random subset |
| **val** (in-training WER) | `dev-clean` | first `VAL_N` utterances **sorted by id**: fixed and deterministic |
| **test** | `test-clean`, `test-other` | everything, only used at the end (`train.py`'s own `[a-z' ]` WER) |
| **benchmark** | Open ASR sets + local `fleurs_en` / `edacc` test sets (`BENCHMARK_SETS`) | test splits only, run by `benchmark.py` after training (Whisper normaliser) |

**Model selection only sees LibriSpeech.** The val split is `dev-clean`, so AMI and FLEURS are in the training mix but
never in the in-training WER; the other benchmark sets (VoxPopuli, Earnings-22, GigaSpeech, SPGISpeech, …) have no
training data here, so on those the model is zero-shot and out of domain.

**The val subset is small and narrow.** The first `VAL_N=200` dev-clean utterances by id come from only **3 of the 40
dev-clean speakers** (8 chapters, ~23 min). It's fixed and deterministic, so curves are comparable across runs, but it
is a noisy estimate of dev-clean and says nothing about "other" or out-of-domain speech.

LibriSpeech splits are already speaker-disjoint; the cell below checks it (LibriSpeech only: the other corpora's ids
don't encode the speaker).

In [ ]:
train_all = [it for name in TRAIN_PARTS for it in splits[name]]
train_items = [it for it in train_all if T.MIN_TRAIN_SEC <= it["sec"] <= T.MAX_TRAIN_SEC]
val_items = sorted(splits[T.VAL_SPLIT], key=lambda it: it["id"])[:T.VAL_N]
dropped = len(train_all) - len(train_items)
print(f"train: {len(train_items)} utts ({sum(it['sec'] for it in train_items) / 3600:.1f} h), "
      f"dropped {dropped} outside [{T.MIN_TRAIN_SEC:g}, {T.MAX_TRAIN_SEC:g}] s")
print(f"val  : {len(val_items)} utts ({sum(it['sec'] for it in val_items) / 60:.1f} min) from "
      f"{len({speaker(it) for it in val_items})} speakers: {val_items[0]['id']} .. {val_items[-1]['id']}")

spk = {name: {speaker(it) for it in items} for name, items in splits.items() if "/" not in name}
names = list(spk)
for i, a in enumerate(names):
    for b in names[i + 1:]:
        print(f"speaker overlap {a:>16} & {b:<16}: {len(spk[a] & spk[b])}")

### Batching: length-bucketed sampler

`T.create_length_batched_sampler()` (new seed each epoch):
1. shuffle all utterance indices
2. cut into chunks of `BATCH_SIZE * 50` and **sort each chunk by duration**
3. slice each sorted chunk into batches of `BATCH_SIZE`, then shuffle the batch order

So each batch holds utterances of similar length (little padding), while the order of batches stays random.
`GRAD_ACCUM` micro-batches make one optimizer step.

In [ ]:
batches = T.create_length_batched_sampler(train_items, T.BATCH_SIZE, seed=T.SEED)
idx = list(range(len(train_items)))
random.Random(0).shuffle(idx)
random_batches = [idx[i:i + T.BATCH_SIZE] for i in range(0, len(idx), T.BATCH_SIZE)]

def pad_efficiency(bs):
    real = sum(train_items[i]["sec"] for b in bs for i in b)
    padded = sum(max(train_items[i]["sec"] for i in b) * len(b) for b in bs)
    return real / padded

steps_per_epoch = len(batches) // T.GRAD_ACCUM
print(f"{len(batches)} micro-batches/epoch of {T.BATCH_SIZE} -> {steps_per_epoch} optimizer steps/epoch "
      f"(x{T.GRAD_ACCUM} accum = eff. batch {T.BATCH_SIZE * T.GRAD_ACCUM}), {steps_per_epoch * T.EPOCHS} total")
print(f"audio padding efficiency: bucketed {pad_efficiency(batches):.1%} vs plain random {pad_efficiency(random_batches):.1%}")
print("\nfirst 3 batches (durations in s):")
for b in batches[:3]:
    print("  ", [round(train_items[i]["sec"], 1) for i in b])

fig, ax = plt.subplots(figsize=(13, 3))
for k, (bs, label) in enumerate([(batches, "length-bucketed"), (random_batches, "random")]):
    for j, b in enumerate(bs[:60]):
        d = [train_items[i]["sec"] for i in b]
        ax.plot([j + 0.3 * k] * len(d), d, ".", color=f"C{k}", ms=4, label=label if j == 0 else None)
ax.set(xlabel="batch index", ylabel="utterance duration (s)", title="Durations inside the first 60 batches")
ax.legend(); plt.show()

## 3. Audio features: how the audio is put into the model

There is **no audio encoder**. Each step below is a line of `T.extract_log_mel_spectrogram()` (reads the file, then
`T.log_mel_from_waveform()`, which `benchmark.py` also uses on decoded/resampled audio) or
`T.SpeechDataset.__getitem__()`; here they are run one at a time so every intermediate can be plotted.

### 3a. STFT → power spectrogram
Window of `N_FFT=400` samples (25 ms), hop of `HOP=160` samples (10 ms) → **`SR / HOP` = 100 frames/s** with the defaults, `N_FFT/2+1 = 201` frequency bins.

In [ ]:
stft = torch.stft(torch.from_numpy(wav), n_fft=T.N_FFT, hop_length=T.HOP, window=T.STFT_WINDOW, return_complex=True)
power = stft.abs() ** 2
print("waveform", tuple(wav.shape), "-> STFT", tuple(stft.shape), "(freq bins, frames)")

plt.figure(figsize=(13, 3))
plt.imshow(10 * np.log10(power.numpy() + 1e-10), origin="lower", aspect="auto", cmap="magma",
           extent=[0, power.shape[1] / FPS, 0, T.SR / 2])
plt.colorbar(label="dB"); plt.xlabel("time (s)"); plt.ylabel("Hz"); plt.title("linear-frequency power spectrogram"); plt.show()

### 3b. Mel filterbank (HTK, 80 triangles) → log
`T.MEL_FILTERBANK` is an `(80, 201)` matrix: each row is a triangle on the linear-frequency axis,
narrow at low frequencies and wide at high frequencies. `log(filterbank @ power + 1e-6)` gives `(frames, 80)`.

In [ ]:
fb = T.MEL_FILTERBANK
freqs = np.fft.rfftfreq(T.N_FFT, 1 / T.SR)
log_mel = T.extract_log_mel_spectrogram(sample["path"])          # the function the Dataset uses
assert torch.allclose(log_mel, torch.log(fb @ power + 1e-6).T)   # == the manual steps above
print("filterbank", tuple(fb.shape), "| log-mel", tuple(log_mel.shape), "(frames, mel bins)")

fig, axes = plt.subplots(1, 2, figsize=(13, 3), gridspec_kw={"width_ratios": [1, 2]})
for m in range(0, T.N_MELS, 3):
    axes[0].plot(freqs, fb[m], lw=0.8)
axes[0].set(xlabel="Hz", title="mel filterbank (every 3rd filter)")
im = axes[1].imshow(log_mel.T, origin="lower", aspect="auto", cmap="magma", extent=[0, log_mel.shape[0] / FPS, 0, T.N_MELS])
axes[1].set(xlabel="time (s)", ylabel="mel bin", title="log-mel"); plt.colorbar(im, ax=axes[1])
plt.tight_layout(); plt.show()

### 3c. Global mean-variance normalisation (MVN)
One mean and std **per mel bin**, computed once from `MVN_N=2000` random utterances of the MVN split (`MVN_SPLIT`,
default: the train mix) and cached at `T.mvn_cache_path()` (the file name encodes the split and every feature knob). After MVN every bin is roughly zero-mean / unit-variance,
so **0 = "average frame"**, which is why the tail is padded with zeros next.

In [ ]:
mvn = T.compute_or_load_mvn_stats(train_items)   # loads the cache
print("MVN stats:", T.mvn_cache_path())
norm_mel = (log_mel - mvn["mean"]) / mvn["std"]

fig, axes = plt.subplots(1, 3, figsize=(14, 3), gridspec_kw={"width_ratios": [1, 1, 2]})
axes[0].plot(mvn["mean"]); axes[0].set(title="MVN mean per bin", xlabel="mel bin")
axes[1].plot(mvn["std"], color="C1"); axes[1].set(title="MVN std per bin", xlabel="mel bin")
im = axes[2].imshow(norm_mel.T, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3,
                    extent=[0, norm_mel.shape[0] / FPS, 0, T.N_MELS])
axes[2].set(xlabel="time (s)", ylabel="mel bin", title="normalised log-mel"); plt.colorbar(im, ax=axes[2])
plt.tight_layout(); plt.show()
print(f"before MVN: mean {log_mel.mean():.2f} std {log_mel.std():.2f} | after: mean {norm_mel.mean():.2f} std {norm_mel.std():.2f}")

### 3d. Pad + stack `STACK` frames → one audio patch
The spectrogram is cut **only along time** into chunks of `STACK` frames (`STACK × HOP / SR` seconds), keeping all
`N_MELS` mel bins. Each `STACK × N_MELS` chunk is flattened row-major into a **`STACK·N_MELS`-d vector**:
`[frame0 bins 0..N_MELS-1, frame1 bins …, …]` (`STACK=16`: 16 × 80 = 1280-d, 160 ms; `STACK=8`: 640-d, 80 ms).
The last chunk is zero-padded if the frame count isn't a multiple of `STACK`.


In [ ]:
pad = (-norm_mel.shape[0]) % T.STACK
padded = torch.cat([norm_mel, norm_mel.new_zeros(pad, T.N_MELS)])
patches = padded.reshape(-1, T.STACK * T.N_MELS)
print(f"{norm_mel.shape[0]} frames + {pad} zero frames = {padded.shape[0]} -> patches {tuple(patches.shape)}")

# Same thing through the Dataset the training loop uses
tok = T.AutoTokenizer.from_pretrained(T.MODEL)
ds = T.SpeechDataset([sample], mvn, tok)
ex = ds[0]
assert torch.equal(ex["audio"], patches)
print("SpeechDataset item:", {k: (tuple(v.shape) if torch.is_tensor(v) else v if isinstance(v, str) else f"list[{len(v)}]")
                              for k, v in ex.items()})
# patch k, frame f, bin b  <->  padded[k*STACK + f, b]  <->  patches[k, f*N_MELS + b]
assert torch.equal(patches[5].view(T.STACK, T.N_MELS), padded[5 * T.STACK:6 * T.STACK])

In [ ]:
# Zoom into the first 2 seconds and draw the patch boundaries
n_show = min(len(patches), max(1, round(2 * FPS / T.STACK)))   # patches in 2 s
fig = plt.figure(figsize=(14, 6.5))
ax = fig.add_subplot(2, 1, 1)
ax.imshow(padded[: n_show * T.STACK].T, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
for k in range(n_show + 1):
    ax.axvline(k * T.STACK - 0.5, color="k", lw=0.8)
for k in range(0, n_show, max(1, n_show // 40)):
    ax.text(k * T.STACK + (T.STACK - 1) / 2, T.N_MELS + 1, str(k), ha="center", fontsize=7)
ax.set(xlabel=f"frame ({1000 / FPS:.0f} ms)", ylabel="mel bin")
ax.set_title(f"normalised log-mel, first {n_show} patches "
             f"(each box = {T.STACK} frames = {1000 * T.STACK / FPS:.0f} ms = 1 LLM token)", pad=14)

k = min(12, n_show - 1)
ax = fig.add_subplot(2, 3, 4)
ax.imshow(patches[k].view(T.STACK, T.N_MELS).T, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
ax.set(title=f"patch {k} as {T.STACK} × {T.N_MELS}", xlabel="frame in patch", ylabel="mel bin")
ax = fig.add_subplot(2, 3, (5, 6))
ax.plot(patches[k], lw=0.8)
for f in range(1, T.STACK):
    ax.axvline(f * T.N_MELS, color="gray", lw=0.6, ls="--")
ax.set(title=f"patch {k} flattened -> {PATCH_DIM}-d vector = 1 audio token (dashed = frame boundaries)", xlabel="dim")
plt.tight_layout(); plt.show()


## 4. Choosing the front-end

Everything in this section is measured **without training**: it shows what each front-end knob costs and what it
throws away, for the current `MODEL`.

### 4a. Patch size vs LLM width

With `STACK=8` a patch has **640 numbers but becomes a 1024-d token** (230M). A `linear` projector *widens* its input, so
`W` has rank ≤ 640 and every audio token lives in a 640-d (affine) subspace of the embedding space: ~40% of each token's
width carries nothing new. (The `mlp` and `conv` projectors are non-linear, so their tokens aren't confined to a
subspace, but each token still summarises only 640 input numbers; `conv` also looks 7 frames into the previous patch.)

Larger patches (`STACK` ↑) put more audio into each token and make the sequence shorter (attention and decode get
cheaper, `PATCH_DIM ≥ hidden_size` lets a linear projection use the full width). The cost is time resolution: each
token has to summarise more speech, and the audio token rate drifts toward the text token rate.
The table below shows the trade-off for the current `MODEL` / `N_MELS` (`audio/text` = audio tokens per transcript token
on the train mix; it should stay well above 1). Try a value with `OVERRIDES = {"STACK": ...}` and
`python train.py --stack ...`.

In [ ]:
sweep_items = random.Random(0).sample(train_items, min(2000, len(train_items)))
sweep_sec = np.array([it["sec"] for it in sweep_items])
sweep_text = np.array([len(tok(" " + it["text"], add_special_tokens=False).input_ids) + 1 for it in sweep_items])
print(f"text: {sweep_text.sum() / sweep_sec.sum():.2f} tokens/s of speech (incl. EOS), "
      f"mean {sweep_text.mean():.0f} tokens/utt | MODEL hidden_size H = {H}\n")

rows = []
for s in sorted(set(STACK_SWEEP) | {T.STACK}):
    audio_tok = np.ceil(sweep_sec * FPS / s)
    d = s * T.N_MELS
    rows.append(dict(stack=s, patch_dim=d, dim_over_H=d / H, rate_hz=FPS / s, ms_per_tok=1000 * s / FPS,
                     audio_tok=audio_tok.mean(), seq_len=(1 + audio_tok + 2 + sweep_text).mean(),
                     audio_per_text=(audio_tok / sweep_text).mean(), linear_params=d * H + H))

print(f"{'STACK':>6}{'patch':>7}{'dim/H':>7}{'Hz':>7}{'ms/tok':>8}{'audio tok':>11}{'seq len':>9}{'audio/text':>12}{'linear':>10}")
for r in rows:
    mark = "  <- current" if r["stack"] == T.STACK else ""
    print(f"{r['stack']:>6}{r['patch_dim']:>7}{r['dim_over_H']:>7.2f}{r['rate_hz']:>7.2f}{r['ms_per_tok']:>8.0f}"
          f"{r['audio_tok']:>11.0f}{r['seq_len']:>9.0f}{r['audio_per_text']:>12.2f}{r['linear_params'] / 1e6:>9.2f}M{mark}")

fig, ax = plt.subplots(figsize=(9, 3.2))
st = [r["stack"] for r in rows]
ax.plot(st, [r["audio_per_text"] for r in rows], "o-", label="audio tokens per text token")
ax.plot(st, [r["dim_over_H"] for r in rows], "s-", label="patch_dim / hidden_size")
ax.axhline(1, color="k", lw=0.8, ls="--")
ax.axvline(T.STACK, color="C3", lw=1, ls=":", label=f"current STACK={T.STACK}")
ax.set(xlabel="STACK (frames per audio token)", title=f"{T.MODEL.split('/')[-1]}, N_MELS={T.N_MELS}")
ax.legend(); plt.show()


### 4b. Front-end sweep: `SR`, `N_FFT`, `HOP`, `N_MELS`, `STACK` side by side

Edit `FRONTEND_SWEEP` in section 0 (each entry overrides some of the five knobs; `MVN_N` stays fixed) and re-run from here.
Every config is measured on the same `FRONTEND_PROBE_N` train utterances, without training anything:

| column | what it measures | good direction |
|---|---|---|
| `win ms` / `Hz/bin` | STFT window length (time blur) and FFT bin spacing (frequency detail), set by `N_FFT / SR` | trade-off |
| `fps` / `tok Hz` / `ms/tok` | mel frames per second (`SR/HOP`) and audio tokens per second (`SR/HOP/STACK`) | trade-off |
| `dead` | mel filters that cover **no FFT bin** (`N_MELS` too high for `N_FFT`): constant inputs, wasted patch dims | 0 |
| `LSD dB` | log-spectral distance between a fine 16 kHz reference spectrum (512-pt, 4 ms hop) and what survives the log-mel, mapped back to that grid. Measures **information lost by `SR / N_FFT / HOP / N_MELS`** (`STACK` is a lossless reshape, so it doesn't change it) | lower |
| `patch` / `dim/H` | numbers per audio token and their ratio to the LLM width (< 1: a linear projector can only fill a `patch`-d subspace) | ≥ 1 |
| `PC95` / `PC99` / `PC99/H` | principal components holding 95 % / 99 % of the variance of the normalised patches: how many dims the patch *really* uses | ≤ H is enough |
| `audio/text` | audio tokens per transcript token (must stay well above 1, the model emits ≤ 1 text token per audio token of context) | ≳ 2 |
| `seq len` / `LLM cost` | mean `[BOS][audio][prompt][text][EOS]` length, and its ratio to the current config (≈ training/decode compute) | lower |

These are **proxies**: they tell you what each knob costs and what it throws away, not the WER. A config is only "good"
once it is trained; the cell after the plots prints the `train.py` command for each one.


In [ ]:
import soxr
import torch.nn.functional as F

FE_KEYS = ["SR", "N_FFT", "HOP", "N_MELS", "STACK"]
BASE = {k: getattr(T, k) for k in FE_KEYS}
SRC_SR = 16000                                                         # LibriSpeech
probe = random.Random(1).sample(train_items, min(FRONTEND_PROBE_N, len(train_items)))
probe_wavs = [sf.read(it["path"], dtype="float32")[0] for it in probe]

def power_per_hz(power, n_fft, sr):
    # |X|^2 / sum(w^2) / sr = power spectral density: comparable across window lengths and sample rates
    return power / (torch.hann_window(n_fft) ** 2).sum() / sr

REF_FFT, REF_HOP = 512, 64                                             # 31 Hz x 4 ms reference grid at 16 kHz
REF_FREQS = np.fft.rfftfreq(REF_FFT, 1 / SRC_SR)
ref_db = [10 * torch.log10(power_per_hz(torch.stft(torch.from_numpy(w), REF_FFT, REF_HOP, window=torch.hann_window(REF_FFT),
                                                   return_complex=True).abs() ** 2, REF_FFT, SRC_SR) + 1e-20)
          for w in probe_wavs]

def mel_to_linear(log_mel):
    """(frames, N_MELS) log-mel -> (N_FFT/2+1, frames) power: each FFT bin gets the filter-weighted average of the
    mean power per filter (i.e. assumes a flat spectrum inside each filter); bins no filter covers get ~0."""
    fb = T.MEL_FILTERBANK
    per_filter = torch.exp(log_mel).T / fb.sum(1, keepdim=True).clamp_min(1e-8)
    return fb.T @ per_filter / fb.sum(0)[:, None].clamp_min(1e-8)

def log_spectral_distance(log_mel, ref, floor_db=80):
    """RMS dB error (per frame, then averaged) of the log-mel mapped back onto the reference grid. Above the config's
    Nyquist there is no information, so those bins are filled with the utterance's mean reference level."""
    rec = 10 * torch.log10(power_per_hz(mel_to_linear(log_mel), T.N_FFT, T.SR) + 1e-20)
    freqs = np.fft.rfftfreq(T.N_FFT, 1 / T.SR)
    n_ref = int((REF_FREQS <= min(T.SR, SRC_SR) / 2).sum())
    rec = F.interpolate(rec[freqs <= SRC_SR / 2][None, None], size=(n_ref, ref.shape[1]), mode="bilinear",
                        align_corners=True)[0, 0]
    rec = torch.cat([rec, ref.mean().expand(ref.shape[0] - n_ref, ref.shape[1])])
    lo = ref.max() - floor_db                                         # ignore detail more than 80 dB below the peak
    return (rec.clamp_min(lo) - ref.clamp_min(lo)).pow(2).mean(0).sqrt().mean().item()

def griffin_lim(power, n_iter=32):
    """Phase-less (N_FFT/2+1, frames) power -> waveform at T.SR, to *hear* what the features keep."""
    win, mag = torch.hann_window(T.N_FFT), power.sqrt()
    length = (mag.shape[1] - 1) * T.HOP
    phase = torch.exp(2j * np.pi * torch.rand(mag.shape, generator=torch.Generator().manual_seed(0)))
    for _ in range(n_iter):
        y = torch.istft(mag * phase, T.N_FFT, T.HOP, window=win, length=length)
        phase = torch.exp(1j * torch.stft(y, T.N_FFT, T.HOP, window=win, return_complex=True).angle())
    return torch.istft(mag * phase, T.N_FFT, T.HOP, window=win, length=length).numpy()

def label(cfg):
    return ", ".join(f"{k}={v}" for k, v in cfg.items()) or "current"

fe_rows, fe_views = [], []
try:
    for cfg in FRONTEND_SWEEP:
        T.apply_config({**BASE, **cfg})
        fps = T.SR / T.HOP
        wavs = probe_wavs if T.SR == SRC_SR else [soxr.resample(w, SRC_SR, T.SR).astype(np.float32) for w in probe_wavs]
        mels = [T.log_mel_from_waveform(w) for w in wavs]
        cat = torch.cat(mels)
        mean, std = cat.mean(0), cat.std(0).clamp_min(1e-5)           # MVN from the probe set (cache untouched)
        probe_patches = torch.cat([F.pad((m - mean) / std, (0, 0, 0, (-len(m)) % T.STACK)).reshape(-1, T.STACK * T.N_MELS)
                                   for m in mels])
        var = torch.linalg.svdvals(probe_patches - probe_patches.mean(0)) ** 2
        pc95, pc99 = (int((var.cumsum(0) / var.sum() < q).sum()) + 1 for q in (0.95, 0.99))
        audio_tok = np.ceil(sweep_sec * fps / T.STACK)
        seq_len = (1 + audio_tok + 2 + sweep_text).mean()
        fe_rows.append(dict(
            label=label(cfg), cfg=cfg, win_ms=1000 * T.N_FFT / T.SR, hz_bin=T.SR / T.N_FFT, fps=fps,
            tok_hz=fps / T.STACK, ms_tok=1000 * T.STACK / fps,
            dead=int(((T.MEL_FILTERBANK > 0).sum(1) == 0).sum()),
            lsd=np.mean([log_spectral_distance(m, r) for m, r in zip(mels, ref_db)]),
            patch=T.STACK * T.N_MELS, pc95=pc95, pc99=pc99, audio_text=(audio_tok / sweep_text).mean(), seq_len=seq_len))

        # the walk-through sample (section 1) under this config: normalised log-mel + Griffin-Lim resynthesis
        w = wav if T.SR == SRC_SR else soxr.resample(wav, SRC_SR, T.SR).astype(np.float32)
        m = T.log_mel_from_waveform(w)
        fe_views.append(dict(norm=(m - mean) / std, fps=fps, stack=T.STACK, n_mels=T.N_MELS, sr=T.SR,
                             audio=griffin_lim(mel_to_linear(m[: int(6 * fps)]))))
finally:
    T.apply_config(BASE)                                              # back to the notebook's config

base_len = next((r["seq_len"] for r in fe_rows if not r["cfg"]), fe_rows[0]["seq_len"])
print(f"{FRONTEND_PROBE_N} probe utterances | LLM {T.MODEL.split('/')[-1]}: H = {H}\n")
print(f"{'config':<34}{'win ms':>7}{'Hz/bin':>7}{'fps':>5}{'tok Hz':>7}{'ms/tok':>7}{'dead':>5}{'LSD dB':>7}"
      f"{'patch':>6}{'dim/H':>6}{'PC95':>6}{'PC99':>6}{'PC99/H':>7}{'audio/text':>11}{'seq len':>8}{'LLM cost':>9}")
for r in fe_rows:
    print(f"{r['label']:<34}{r['win_ms']:>7.0f}{r['hz_bin']:>7.0f}{r['fps']:>5.0f}{r['tok_hz']:>7.2f}{r['ms_tok']:>7.0f}"
          f"{r['dead']:>5}{r['lsd']:>7.2f}{r['patch']:>6}{r['patch'] / H:>6.2f}{r['pc95']:>6}{r['pc99']:>6}{r['pc99'] / H:>7.2f}"
          f"{r['audio_text']:>11.2f}{r['seq_len']:>8.0f}{r['seq_len'] / base_len:>8.2f}x")


In [ ]:
# Same 3 s of the walk-through sample under every config: what the projector sees (black lines = token boundaries)
n = len(fe_views)
fig, axes = plt.subplots(n, 1, figsize=(13, 1.6 * n), sharex=True)
for ax, v, r in zip(np.atleast_1d(axes), fe_views, fe_rows):
    x = v["norm"][: int(3 * v["fps"])]
    ax.imshow(x.T, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3, extent=[0, len(x) / v["fps"], 0, v["n_mels"]])
    for t in np.arange(0, len(x), v["stack"]) / v["fps"]:
        ax.axvline(t, color="k", lw=0.4)
    ax.set_ylabel("mel bin", fontsize=8)
    ax.set_title(f"{r['label']}  |  {v['n_mels']} mels to {v['sr'] // 2} Hz, {r['fps']:.0f} fps, "
                 f"{r['tok_hz']:.2f} tokens/s, LSD {r['lsd']:.1f} dB", fontsize=9, loc="left")
axes[-1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

# Summary: what each config loses (LSD), how much it asks of the LLM (cost, audio/text) and how it fills the width
fig, axes = plt.subplots(1, 4, figsize=(15, 0.45 * n + 1.5), sharey=True)
y = np.arange(n)
for ax, (key, title, ref) in zip(axes, [
        ("lsd", "LSD vs 16 kHz reference (dB, lower = keeps more)", None),
        ("audio_text", "audio tokens per text token", 1),
        (lambda r: r["seq_len"] / base_len, "LLM cost vs current (x)", 1),
        (lambda r: r["pc99"] / H, "PC99 / hidden size", 1)]):
    vals = [key(r) if callable(key) else r[key] for r in fe_rows]
    ax.barh(y, vals, color=["C3" if not r["cfg"] else "C0" for r in fe_rows])
    if ref is not None:
        ax.axvline(ref, color="k", lw=0.8, ls="--")
    ax.set_title(title, fontsize=9)
axes[0].set_yticks(y, [r["label"] for r in fe_rows], fontsize=8); axes[0].invert_yaxis()
plt.tight_layout(); plt.show()


In [ ]:
# Listen: Griffin-Lim resynthesis of the first 6 s from each config's log-mel (no phase, so it sounds robotic in every
# config; compare intelligibility and muffling between them, e.g. SR=8000 loses the "s"/"f" hiss, N_MELS=40 blurs vowels)
print("original"); display(Audio(wav[: 6 * SRC_SR], rate=SRC_SR))
for v, r in zip(fe_views, fe_rows):
    print(r["label"]); display(Audio(v["audio"], rate=v["sr"]))

print("\nTrain a config (then compare val / test WER in runs/INDEX.md):")
for r in fe_rows:
    flags = " ".join(f"--{k.lower()} {v}" for k, v in r["cfg"].items())
    note = "   # not trainable yet: train.py asserts file SR == SR (LibriSpeech is 16 kHz, no resampling)" \
           if r["cfg"].get("SR", SRC_SR) != SRC_SR else ""
    print(f"  python train.py {flags}".rstrip() + note)


**Reading the sweep.** Each knob moves the proxies in a predictable way:

* **`STACK`** loses nothing (same LSD), it only trades sequence length (cost, `audio/text`) against how much speech
  one token has to summarise. Keep `audio/text` comfortably above ~2 and `patch ≳ H`.
* **`N_MELS`** ↑ lowers LSD (finer formants/pitch harmonics) but grows the patch; check `dead` stays 0 (with
  `N_FFT=400` the lowest filters start to be empty around 128+ mels, then raise `N_FFT` too). `PC95` shows whether the
  extra dims carry anything new.
* **`N_FFT`** ↑ sharpens frequency (`Hz/bin`) but blurs fast events (`win ms`); ↓ the opposite. 20–32 ms is the usual range.
* **`HOP`** ↑ halves the frame rate: cheaper at a fixed `STACK`, or the same token rate with a smaller `STACK`; but
  plosives/short phones get smeared (LSD ↑). Keep `N_FFT ≥ HOP` (ideally ~2.5x) so no audio falls between windows.
* **`SR`** below 16 kHz removes the band above `SR/2` (fricatives); above 16 kHz adds nothing for LibriSpeech (it is
  16 kHz audio) and only spends mel bins on an empty band. Note `train.extract_log_mel_spectrogram` asserts the file
  rate equals `SR`, so training at another `SR` needs a resampling step there first (the sweep resamples with soxr).

To retrain on a front-end, run its printed command; then set `OVERRIDES` in section 0 to the same values to walk the
rest of this notebook through it (that also builds and caches its full `MVN_N` stats in section 3c).


## 5. The projector: audio patch → LLM token

There is **no pretrained audio encoder**. `PROJECTOR` picks a small adapter (`T.build_audio_projector()`), trained from
scratch together with LoRA; its weights are in `runs/<id>/proj.pt`:

| `PROJECTOR` | layers | each token sees |
|---|---|---|
| `linear` | `Linear(STACK·N_MELS → H)` | its own patch |
| `mlp` | `LayerNorm → Linear(STACK·N_MELS → H) → GELU → Linear(H → H)` | its own patch |
| `conv` | `log2(STACK)` strided conv stages (kernel 3, stride 2, GELU) over the *continuous* mel, then `LayerNorm → Linear(H → H) → GELU → Linear(H → H)` | its patch + the `STACK − 1` frames before it |

`PROJ_INIT="match_emb"` initialises the last `Linear` so audio tokens start at the scale of the LLM's text embeddings.
All contextual modelling *across* tokens is left to the LLM. That is why the LLM has to be adapted too: it is frozen,
and LoRA (`LORA_R=64`, `LORA_ALPHA=128`) is added to the LLM's linear layers selected by `LORA_TARGETS` (default
`"all-linear"`). On the hybrid LFM2 stack that is:

| block | layers (230M: 14) | LoRA on |
|---|---|---|
| short-conv blocks | 8 | `in_proj`, `out_proj` |
| attention blocks | 6 | `q_proj`, `k_proj`, `v_proj`, `out_proj` |
| MLP (every block) | 14 | `w1`, `w2`, `w3` |
| embeddings, `lm_head` | | none (frozen) |

`--lora_targets q_proj,k_proj,v_proj,out_proj` restricts it (e.g. attention only); section 7 prints the actual placement
via `T.describe_lora_placement()`, the same line `train.py` logs at startup.

The conv projector, layer by layer on a single token: [`projector_anatomy.ipynb`](projector_anatomy.ipynb).

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors import safe_open
import glob

cfg = AutoConfig.from_pretrained(T.MODEL)
# The LLM's text embedding table, read straight from the checkpoint's safetensors (no need to load the whole model)
model_dir = Path(hf_hub_download(T.MODEL, "config.json")).parent
emb = None
for f in glob.glob(str(model_dir / "*.safetensors")):
    with safe_open(f, "pt") as st:
        for key in st.keys():
            if "embed_tokens" in key:
                emb = st.get_tensor(key).float()
print(f"LLM {T.MODEL}: hidden_size={H}, layers={cfg.num_hidden_layers}, vocab={cfg.vocab_size}, "
      f"text embeddings {tuple(emb.shape)}")

# build_audio_projector() puts the projector on T.DEVICE in the training dtype; here it runs on CPU in fp32
_device, _precision = T.DEVICE, T.PRECISION
T.DEVICE, T.PRECISION = "cpu", "fp32"
try:
    adapter = T.build_audio_projector(H, emb.std().item())
finally:
    T.DEVICE, T.PRECISION = _device, _precision
print(f"\nPROJECTOR={T.PROJECTOR!r}: {sum(p.numel() for p in adapter.parameters()):,} params")
print(adapter)

proj_path = RUN_DIR / "proj.pt" if RUN_DIR else None
TRAINED = proj_path is not None
if TRAINED:
    adapter.load_state_dict({k: v.float() for k, v in torch.load(proj_path, map_location="cpu").items()})
    print("loaded trained projector from", proj_path)
else:
    print("no trained checkpoint found: showing the random init")

with torch.no_grad():
    audio_tokens = adapter(patches)
print(f"\npatches {tuple(patches.shape)} --projector--> audio tokens {tuple(audio_tokens.shape)}  "
      f"({sample['sec']:.2f} s of audio = {audio_tokens.shape[0]} LLM tokens)")

### 5a. What did the trained projector learn? (`linear` / `mlp`)
The first `Linear` of these projectors sees the flattened patch. Reshape its weight from `(hidden, STACK·N_MELS)` to
`(hidden, STACK frames, N_MELS bins)`: the column norms show how much each (frame-in-patch, mel bin) input contributes.
The singular values show how much of the `min(STACK·N_MELS, hidden)` possible rank it actually uses.
(The `conv` projector's first layer sees 3 frames at a time instead; see `projector_anatomy.ipynb`.)

In [ ]:
patch_linear = next((m for m in adapter.modules() if isinstance(m, torch.nn.Linear) and m.in_features == PATCH_DIM), None)
if patch_linear is None:
    print(f"PROJECTOR={T.PROJECTOR!r} has no Linear over the whole patch: see projector_anatomy.ipynb")
else:
    W = patch_linear.weight.detach().view(-1, T.STACK, T.N_MELS)
    col_norm = W.norm(dim=0)                         # (STACK, N_MELS)
    s = torch.linalg.svdvals(patch_linear.weight.detach())

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
    im = axes[0].imshow(col_norm.T, origin="lower", aspect="auto", cmap="viridis")
    axes[0].set(title="||W[:, frame, bin]||", xlabel="frame in patch", ylabel="mel bin"); plt.colorbar(im, ax=axes[0])
    axes[1].plot(col_norm.mean(0)); axes[1].set(title="input weight norm per mel bin (avg over frames)", xlabel="mel bin")
    axes[2].semilogy(s.numpy()); axes[2].set(title=f"singular values of W ({min(W.shape[0], PATCH_DIM)} max rank)", xlabel="index")
    plt.tight_layout(); plt.show()
    print(f"effective rank (singular values > 1% of max): {(s > 0.01 * s[0]).sum().item()} / {len(s)}")

### 5b. Do audio tokens live where text tokens live?
Compare the norms of the projected audio tokens with the LLM's text token embeddings (loaded above).

In [ ]:
text_ids = tok(" " + sample["text"], add_special_tokens=False).input_ids
text_norms = emb[text_ids].norm(dim=1)
audio_norms = audio_tokens.norm(dim=1)
plt.figure(figsize=(8, 3))
plt.hist(emb.norm(dim=1).numpy(), bins=80, alpha=0.5, density=True, label="all vocab embeddings")
plt.hist(text_norms.numpy(), bins=30, alpha=0.6, density=True, label="this transcript's tokens")
plt.hist(audio_norms.numpy(), bins=30, alpha=0.6, density=True, label="this utterance's audio tokens")
plt.xlabel("L2 norm"); plt.legend(); plt.title("embedding norms" + (" (trained projector)" if TRAINED else " (random projector)")); plt.show()
print(f"audio tokens: {len(audio_norms)} for {sample['sec']:.1f} s | text tokens: {len(text_ids)} "
      f"| ratio {len(audio_norms) / len(text_ids):.1f} audio tokens per text token")

## 6. The final input sequence

`T.build_multimodal_inputs()` builds `inputs_embeds` directly (no token ids for audio):

```
[BOS] [audio_0 … audio_N-1] [“Transcription:”] [ text tokens … ] [EOS]
  └────────── labels = -100 (no loss) ──────────┘ └── loss here ────┘
```

* **train**: right-padded with the pad-token embedding, labels `-100` on BOS/audio/prompt/pad
* **generate**: no target, **left-padded** so every row ends at the prompt and generation continues from there

In [ ]:
ds = T.SpeechDataset(train_items, mvn, tok)
# A real bucketed batch has near-identical lengths (almost no padding). To make padding visible,
# take 3 utterances of clearly different lengths (~4 s, ~9 s, ~14 s) instead.
pick = [min(range(len(train_items)), key=lambda i: abs(train_items[i]["sec"] - s)) for s in (4, 9, 14)]
batch = [ds[i] for i in pick]
prompt_ids = tok(T.PROMPT, add_special_tokens=False).input_ids
print("prompt", repr(T.PROMPT), "->", prompt_ids, [tok.decode([i]) for i in prompt_ids])
print("BOS", tok.bos_token_id, repr(tok.bos_token), "| EOS", tok.eos_token_id, repr(tok.eos_token),
      "| PAD", tok.pad_token_id, repr(tok.pad_token))
print("target of item 0:", batch[0]["target"][:12], "...", [tok.decode([i]) for i in batch[0]["target"][:8]], "...")

class EmbOnly(torch.nn.Module):
    # build_multimodal_inputs only calls model.get_input_embeddings(); avoid loading the full LLM here
    def __init__(self, w): super().__init__(); self.e = torch.nn.Embedding.from_pretrained(w)
    def get_input_embeddings(self): return self.e

emb_model = EmbOnly(emb)
# This illustration uses the CPU fp32 embedding table + projector from above (exact checks in the next cell);
# build_multimodal_inputs reads T.DEVICE / T.DTYPE, so switch them to CPU fp32 just for these two calls.
_device, _dtype = T.DEVICE, T.DTYPE
T.DEVICE, T.DTYPE = "cpu", torch.float32
try:
    x_tr, m_tr, y_tr = T.build_multimodal_inputs(batch, emb_model, adapter, tok, prompt_ids, is_train=True)
    x_ge, m_ge, _ = T.build_multimodal_inputs(batch, emb_model, adapter, tok, prompt_ids, is_train=False)
finally:
    T.DEVICE, T.DTYPE = _device, _dtype
print(f"\ntrain : inputs_embeds {tuple(x_tr.shape)}, attention_mask {tuple(m_tr.shape)}, labels {tuple(y_tr.shape)}")
print(f"decode: inputs_embeds {tuple(x_ge.shape)}")
for it, row in zip(batch, m_tr):
    print(f"  {it['id']}: 1 BOS + {it['audio'].shape[0]} audio + {len(prompt_ids)} prompt + {len(it['target'])} target"
          f" = {int(row.sum())}  (pad {row.numel() - int(row.sum())})")

In [ ]:
from matplotlib.colors import ListedColormap
KIND = ["pad", "BOS", "audio", "prompt", "target / EOS"]
cmap = ListedColormap(["#dddddd", "#000000", "#4c72b0", "#dd8452", "#55a868"])

def layout(batch, n_cols, left):
    grid = np.zeros((len(batch), n_cols), dtype=int)
    for r, it in enumerate(batch):
        seg = [1] + [2] * it["audio"].shape[0] + [3] * len(prompt_ids) + ([4] * len(it["target"]) if not left else [])
        start = n_cols - len(seg) if left else 0
        grid[r, start:start + len(seg)] = seg
    return grid

fig, axes = plt.subplots(3, 1, figsize=(14, 5.5), sharex=False)
axes[0].imshow(layout(batch, x_tr.shape[1], False), aspect="auto", cmap=cmap, vmin=0, vmax=4, interpolation="nearest")
axes[0].set_title("training sequence (right-padded)")
axes[1].imshow((y_tr != -100).numpy(), aspect="auto", cmap="Greens", interpolation="nearest")
axes[1].set_title("positions with a loss (labels != -100)")
axes[2].imshow(layout(batch, x_ge.shape[1], True), aspect="auto", cmap=cmap, vmin=0, vmax=4, interpolation="nearest")
axes[2].set_title("generation input (left-padded, no target)")
for ax in axes: ax.set_yticks(range(len(batch))); ax.set_ylabel("item")
axes[2].set_xlabel("sequence position")
handles = [plt.Rectangle((0, 0), 1, 1, color=cmap(i)) for i in range(5)]
fig.legend(handles, KIND, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 1.03))
plt.tight_layout(); plt.show()

# sanity: embedding rows match their sources
r = 0; n_a = batch[r]["audio"].shape[0]
assert torch.allclose(x_tr[r, 0], emb[tok.bos_token_id])
assert torch.allclose(x_tr[r, 1:1 + n_a], adapter(batch[r]["audio"]), atol=1e-5)
assert torch.allclose(x_tr[r, 1 + n_a:1 + n_a + len(prompt_ids)], emb[prompt_ids])
print("row 0 checks passed: BOS / audio / prompt embeddings are where expected")

## 7. (optional) Forward pass and greedy decode with the trained checkpoint

Loads the LLM + LoRA on `DEVICE` and prints where LoRA sits (230M: ~1 GB in bf16). Only runs if a matching run was
found (`RUN_DIR` in section 0) and `DEVICE` is a GPU: LFM2's short-conv blocks use a CUDA-only kernel.


In [ ]:
RUN_LLM = TRAINED and DEVICE != "cpu"
if TRAINED and not RUN_LLM:
    print("skipped: LFM2's short-conv kernel needs a GPU (DEVICE = 'cpu')")

if RUN_LLM:
    tok_, model, proj = T.load_model_and_projector(str(RUN_DIR))
    model.eval()
    print(T.describe_lora_placement(model))
    with torch.no_grad():
        x, m, y = T.build_multimodal_inputs(batch, model, proj, tok, prompt_ids, is_train=True)
        out = model(inputs_embeds=x, attention_mask=m, labels=y)
        print(f"teacher-forced loss on 3 items: {out.loss.item():.3f}  (logits {tuple(out.logits.shape)})")

        x, m, _ = T.build_multimodal_inputs(batch, model, proj, tok, prompt_ids, is_train=False)
        gen = model.generate(inputs_embeds=x, attention_mask=m, max_new_tokens=60, do_sample=False,
                             eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    for it, g in zip(batch, gen):
        print(f"\n[{it['id']}]\nREF: {it['text']}\nHYP: {T.normalize_text_for_wer(tok.decode(g, skip_special_tokens=True))}")

## 8. Run tracking: finding runs and their plots

Every `python train.py` gets its own `runs/<YYYYmmdd-HHMMSS>_<model>_stack<N>[_<changed hparams>]/` folder, e.g.
`--epochs 10 --batch_size 16` → `..._stack16_epochs10_bs16`.

* `runs/latest` → symlink to the newest run (`python runs.py latest`, `python runs.py show latest`)
* `runs/INDEX.md` → every run, newest first: status/step, what it changed, best val, test and benchmark WER
* `runs/<id>/plots.png` → loss, grad norm vs clip, lr, val WER (+sub/del/ins), throughput, memory; redrawn every
  `PLOT_EVERY` steps and after each val (`python runs.py plot <run>` for older runs)
* after training, `train.py` runs `benchmark.py` on all Open ASR sets (`BENCHMARK_SETS`) and stores the WERs in
  `runs/<id>/summary.json` under `benchmark`


In [ ]:
from IPython.display import Image, Markdown
import runs as R

latest = R.latest_run_dir()
print("latest run:", latest)
display(Markdown(Path("runs/INDEX.md").read_text() if Path("runs/INDEX.md").exists() else "no runs/INDEX.md yet"))
if latest is not None and R.plot_run(latest):   # (re)draw it now so it is current
    display(Image(filename=str(latest / "plots.png")))
